# PAA KX2

The [PAA KX2](https://www.peakanalysisandautomation.com/products/plate-handlers/kx2/) (also sold as KiNEDx KX2) is a 5-axis SCARA plate-handling robot from Peak Analysis and Automation. It supports:

- [Arms](../../capabilities/arms) (pick/place, Cartesian and joint movement, freedrive teaching)

The device communicates over CAN bus using the CANopen protocol (CiA-301 + CiA-402 drive profile). A PCAN/SocketCAN-compatible CAN interface is required (e.g. PEAK PCAN-USB).

| Model | PLR Name | Gripper | Notes |
|---|---|---|---|
| KX2 | `KX2` | servo | 4 motion axes (shoulder, Z, elbow, wrist) + servo gripper |

## Setup

`setup()` opens the CAN bus, discovers nodes, reads drive parameters, enables the motion axes, and homes + closes the servo gripper.

In [ ]:
from pylabrobot.paa.kx2 import KX2

kx2 = KX2()
await kx2.setup()

## Arm capabilities

The KX2 exposes an {class}`~pylabrobot.capabilities.arms.orientable_arm.OrientableArm` on `kx2.arm`, backed by {class}`~pylabrobot.paa.kx2.KX2ArmBackend`. Gripper yaw is controlled via a single `direction` float (degrees; 0° = front). For the full arm API, see [Arms](../../capabilities/arms).

The raw driver (`kx2.driver`, a `KX2Driver`) stays available for low-level access — motor commands, drive diagnostics, binary interpreter, etc.

### Gripper control

The servo gripper is force-aware. `close_gripper` moves to the target width and then verifies the plate is gripped; pass `check_plate_gripped=False` via `GripParams` to skip the verification move. Units are KX2 gripper encoder units (mm-equivalent once calibrated).

In [ ]:
await kx2.arm.open_gripper(gripper_width=30)

In [ ]:
from pylabrobot.paa.kx2 import KX2ArmBackend

await kx2.arm.close_gripper(
    gripper_width=0,
    backend_params=KX2ArmBackend.GripParams(check_plate_gripped=False),
)

In [ ]:
await kx2.arm.is_gripper_closed()

Cap the gripping force on a per-close basis (as a percentage of the motor's peak current — 10..100, clamped) when closing on fragile labware:

In [ ]:
await kx2.arm.close_gripper(
    gripper_width=0,
    backend_params=KX2ArmBackend.GripParams(max_force_percent=30),
)

### Proximity sensor

An IR breakbeam between the gripper fingers, wired to the Z drive's IO header. The sensor is binary (no distance, just present/absent) and is exposed via the auto-discovered `"ProximitySensor"` digital input on the Z axis (pin 4).

`read_proximity_sensor()` returns `True` when the beam is interrupted. Useful for confirming an object is in the gripper before closing.

In [ ]:
await kx2.arm.backend.read_proximity_sensor()

`find_z_with_proximity_sensor` descends Z up to `max_descent` and halts the instant the beam trips. It arms the Elmo drive's input-logic register so the drive itself stops the motor on the input edge (sub-ms reaction; no software in the loop). Returns the Z where the drive halted; raises `RuntimeError` if the beam never tripped.

Pass `z_start` to first move Z to a known starting height before searching. Position the arm above an object and open the gripper wider than the object before running.

In [ ]:
z_hit = await kx2.arm.backend.find_z_with_proximity_sensor(max_descent=100.0, z_start=400)
print(f"object found at Z = {z_hit:.2f}")

### Motion limits

`motion_limits()` reads the per-axis firmware maxima cached at setup. `JointMoveParams` / `CartesianMoveParams` take `max_gripper_speed` (mm/s) and `max_gripper_acceleration` (mm/s^2), which cap the worst-case Cartesian speed/acceleration at the gripper along the trajectory. `None` (the default) means no Cartesian cap — joints run at firmware max. Joint speeds get scaled uniformly so the gripper stays under the cap.

In [ ]:
kx2.arm.backend.motion_limits()

### Cartesian movement

Move the arm to a Cartesian location. `direction` is the gripper yaw in degrees (Z-axis rotation only — the KX2 cannot roll or pitch). Use {class}`~pylabrobot.paa.kx2.KX2ArmBackend.CartesianMoveParams` to override velocity and acceleration.

In [ ]:
await kx2.arm.request_gripper_location()

In [ ]:
from pylabrobot.resources import Coordinate

await kx2.arm.move_to_location(
    location=Coordinate(x=330.124, y=210.552, z=320.0441),
    direction=198,
    backend_params=KX2ArmBackend.CartesianMoveParams(
        max_gripper_speed=200.0, max_gripper_acceleration=1000.0,
    ),
)

### Joint movement

Move in joint space using the {class}`~pylabrobot.paa.kx2.KX2ArmBackend.Axis` enum. Rotary axes in degrees; Z (and gripper) in mm-equivalent encoder units.

```{warning}
Moving to arbitrary joint positions can cause the arm to collide with its base, gripper, or nearby equipment. Verify coordinates in a safe workspace first, and start with low velocity.
```

In [ ]:
from pylabrobot.paa.kx2 import KX2ArmBackend

position = {
    KX2ArmBackend.Axis.SHOULDER: 0.0,
    KX2ArmBackend.Axis.Z: 150.0,
    KX2ArmBackend.Axis.ELBOW: 90.0,
    KX2ArmBackend.Axis.WRIST: 0.0,
}
await kx2.arm.backend.move_to_joint_position(
    position,
    backend_params=KX2ArmBackend.JointMoveParams(
        max_gripper_speed=200.0, max_gripper_acceleration=1000.0,
    ),
)

### Joint-space pick and place

`pick_up_at_joint_position` / `drop_at_joint_position` do the same thing as their Cartesian counterparts (move, then close/open the gripper), but the target pose is specified in joint space. Useful when you've taught a position via freedrive and want to return to it without going through inverse kinematics.

In [ ]:
pick_joints = {
    KX2ArmBackend.Axis.SHOULDER: 0.0,
    KX2ArmBackend.Axis.Z: 150.0,
    KX2ArmBackend.Axis.ELBOW: 90.0,
    KX2ArmBackend.Axis.WRIST: 0.0,
}
place_joints = {
    KX2ArmBackend.Axis.SHOULDER: 30.0,
    KX2ArmBackend.Axis.Z: 150.0,
    KX2ArmBackend.Axis.ELBOW: 90.0,
    KX2ArmBackend.Axis.WRIST: 0.0,
}

await kx2.arm.backend.pick_up_at_joint_position(pick_joints, resource_width=0)
await kx2.arm.backend.drop_at_joint_position(place_joints, resource_width=30)

### Querying position

Read the current joint angles or Cartesian end-effector pose:

In [ ]:
await kx2.arm.backend.request_joint_position()

In [ ]:
await kx2.arm.request_gripper_location()

### Pick and place

`pick_up_at_location` moves to the target pose and closes the gripper to `resource_width`. `drop_at_location` moves and opens the gripper. Approach and retract motions are the caller's responsibility — bracket these calls with your own pre- and post-moves.

In [ ]:
pick  = Coordinate(x=450.0, y=0.0,    z=80.0)
place = Coordinate(x=450.0, y=-200.0, z=80.0)
above = Coordinate(x=0,     y=0,      z=60.0)
yaw   = 0.0

# pick_up_at_location stores resource_width on the frontend; drop_at_location
# reuses it, so you don't pass it again.
await kx2.arm.move_to_location(pick + above, direction=yaw)
await kx2.arm.pick_up_at_location(location=pick, direction=yaw, resource_width=0)
await kx2.arm.move_to_location(pick + above, direction=yaw)

await kx2.arm.move_to_location(place + above, direction=yaw)
await kx2.arm.drop_at_location(location=place, direction=yaw)
await kx2.arm.move_to_location(place + above, direction=yaw)

### Freedrive (teaching mode)

Freedrive disables torque on the motion axes so you can push the arm by hand; the servo gripper stays powered. The KX2 frees all motion axes at once — the `free_axes` argument is accepted for API parity and ignored.

In [ ]:
await kx2.arm.backend.start_freedrive_mode(free_axes=[0])

In [ ]:
# Manually guide the arm to the desired pose, then read it:
taught = await kx2.arm.request_gripper_location()
print(taught)

In [ ]:
await kx2.arm.backend.stop_freedrive_mode()

### Halt and fault diagnostics

`halt` issues an emergency stop on every motion axis. Driver-level methods give you estop-line state and per-axis fault codes.

In [ ]:
await kx2.arm.backend.halt()

In [ ]:
await kx2.driver.get_estop_state()

In [ ]:
from pylabrobot.paa.kx2 import KX2ArmBackend

await kx2.driver.motor_get_fault(KX2ArmBackend.Axis.SHOULDER)

## Barcode reader

The KX2's onboard barcode reader is a serial 1D laser scanner wired to the controller PC. It's fully independent of the CAN motor stack, so it works even with the arm e-stopped — exposed as its own {class}`~pylabrobot.paa.kx2.KX2BarcodeReader` `Device`.

The model that ships with KX2 systems we've inspected is a **Denso MDI-4050** (firmware reports as `BD01J09`). Factory defaults: **9600 8N1**, no flow control. Find your port with `python -m serial.tools.list_ports -v`.

### macOS: USB-to-serial driver

The reader cable is typically a Prolific PL2303. macOS bundles a driver for older variants but not the newer GC/HXN chip (USB ID `067b:23a3`). If the device doesn't show up at `/dev/tty.PL2303G-USBtoUART<n>` after plugging it in:

```bash
brew install --cask prolific-pl2303
open -a PL2303Serial      # registers the system extension
```

Then **System Settings → Privacy & Security → Allow** on the "System software from PL2303Serial was blocked" prompt, restart if asked, and replug the cable. Verify with `systemextensionsctl list`: `com.prolific.cdc.PLCdcFSDriver` should show `[activated enabled]`.

### Symbologies

The MDI-4050 is a 1D laser, so **PDF417 / DataMatrix / QR Code aren't supported** — you'd need the 2D imager variants (e.g. MDI-4150) for those.

Typical out-of-the-box 1D enables: UPC-A, UPC-E, EAN-13, EAN-8, Code 39, Tri-Optic, Codabar, Industrial 2/5, Interleaved 2/5, IATA, Code 128, Code 93, GS1 DataBar, GS1 DataBar Limited. Add-on (supplemental) codes, postal codes, MSI/Plessey, Telepen, UK/Plessey, and Code 11 are off by default.

You can read your unit's full configuration over the wire — model, firmware, every per-symbology enable flag, prefixes/suffixes, and length limits — via `dump_config()` (which sends `Z4`):

```python
text = await bcr.driver.dump_config()
print(text[:200])  # header has MODEL / ROM Ver. / I/F
```

To **change** which symbologies are enabled, three options in order of practicality:

1. **Denso's MDI Setup utility** (free Windows download from Denso ADC) — checkbox UI, talks to the reader over the same serial port, writes to NVRAM.
2. **Configuration barcodes** printed in the MDI-4050 setting manual — scan a Start → toggle → End sequence; no host required.
3. Direct register writes via the ESC protocol — possible but the per-symbology byte layout is documented only in Denso's protocol manual.

{class}`~pylabrobot.resources.barcode.Barcode` returned from `scan()` has `symbology="ANY 1D"` by default — the reader doesn't emit a symbology ID unless prefix/suffix bytes are configured to include one.

### Setup

Pass the serial port path. `setup()` opens the port, does the `Z1` software-version handshake, and puts the reader into single-trigger mode.

In [ ]:
from pylabrobot.paa.kx2 import KX2BarcodeReader

bcr = KX2BarcodeReader(port="/dev/tty.PL2303G-USBtoUART11240")
await bcr.setup()

### Scan

`scan(read_time=...)` fires the trigger and waits up to `read_time + 1s` for the reader to decode something. Omit `read_time` to use whatever read window is currently programmed on the reader. Returns a {class}`~pylabrobot.resources.barcode.Barcode`.

In [ ]:
barcode = await bcr.barcode_scanning.scan(read_time=8)
print(barcode.data)

### Configuration

Change the trigger mode at runtime:

In [ ]:
# "single" (default), "multiple" (two reads per trigger), or "continuous".
# await bcr.set_read_mode("continuous")

### Driver-level access

For raw vendor commands or the auto-trigger mode (reader fires on its own), go through `bcr.driver`:

In [ ]:
print(await bcr.driver.get_software_version())

# Full NVRAM dump — model, firmware, all symbology enables, prefixes/suffixes, length limits.
config = await bcr.driver.dump_config()
print(config[:200])

# await bcr.driver.set_auto_trigger(True)   # reader scans on its own
# await bcr.driver.set_auto_trigger(False)

### Teardown

`stop()` sends `Y` (trigger off) and `Y2` (reset to a 2s read window), then closes the serial port — leaving the reader in a known state for the next session.

In [ ]:
await bcr.stop()

## Teardown

In [ ]:
await kx2.stop()